In [20]:
import pandas as pd
import html


# Proyecto: Análisis Multidimensional de Fenómenos Anómalos No Identificados (FANI)

Analistas: 

Marianela

Caudia


* Herramientas: Python (Pandas) para ETL + SAS Viya para Analítica Avanzada.

## 1. Introducción y Marco Hipotético
El presente proyecto propone un enfoque cuantitativo y relacional sobre los reportes de avistamientos FANI/OVNI. El objetivo es analizar el entorno en el que ocurren los reportes: ¿Cómo afecta el clima, la fase lunar o la proximidad a un aeropuerto en la probabilidad de un reporte?

### Hipótesis de Trabajo:

* **Sesgo Geográfico**: Los reportes presentan una concentración en países anglosajones y zonas de alta densidad poblacional.

* **Morfología vs. Duración**: Existe una relación entre la forma del objeto y su tiempo de visibilidad.

* **Factor Visibilidad**: La mayoría de los avistamientos ocurren bajo condiciones de cielo despejado.

* **Interferencia Aeronáutica**: Una proporción de los casos se debe a la interpretación errónea de tráfico aéreo comercial.

* **Estacionalidad**: El fenómeno presenta picos en meses de verano (mayor actividad al aire libre).

* **Réplica Local (Argentina)**: Los casos locales siguen las mismas tendencias ambientales que el modelo global.

## 2. Carga y Filtrado Inicial (Optimización para SAS Viya)
En este paso cargamos el dataset ambiental (Hugging Face) y realizamos un filtro temporal desde el año 2005. El objetivo es asegurar la relevancia estadística reciente y reducir el peso del archivo para cumplir con el límite de 100MB de la plataforma académica.

In [24]:
# Carga de datos originales
df_ambiental = pd.read_json("ufo_data_clustered.jsonl", lines=True)

# Filtro temporal: registros posteriores al 01/01/2005
df_ambiental['t_utc'] = pd.to_datetime(df_ambiental['t_utc'], errors='coerce')
df_limpio = df_ambiental[df_ambiental['t_utc'].dt.year >= 2005].copy()

# Eliminación de columnas redundantes y pesadas (texto y metadatos del cluster)
columnas_a_eliminar = ['text', 'uid', 'src', 'reports_z', 'nearest_airport_code', 'prob']
df_limpio = df_limpio.drop(columns=columnas_a_eliminar)

print(f"Dataset optimizado: {len(df_limpio)} filas.")

Dataset optimizado: 192777 filas.


## 3. Enriquecimiento de Datos (Fuzzy Matching Join)
Para recuperar las variables 'shape' (forma) y 'duration' (duración), realizamos un cruce relacional (Left Join) con el dataset histórico de Kaggle. Debido a inconsistencias en los registros de segundos, implementamos una 'llave relajada' basada en la fecha (Y-M-D) y el nombre de la ciudad normalizado.

In [25]:
# Carga del dataset secundario para rescate de variables
df_original = pd.read_csv("scrubbed.csv", low_memory=False)
df_rescate = df_original[['datetime', 'city', 'shape', 'duration (seconds)']].copy()

# Normalización de fechas y creación de llaves para el cruce
df_rescate['datetime'] = df_rescate['datetime'].str.replace('24:00', '00:00')
df_rescate['t_utc'] = pd.to_datetime(df_rescate['datetime'], errors='coerce')

df_limpio['t_utc'] = df_limpio['t_utc'].dt.tz_localize(None) # Quitar zona horaria
df_limpio['fecha_corta'] = df_limpio['t_utc'].dt.date
df_rescate['fecha_corta'] = df_rescate['t_utc'].dt.date

df_limpio['ciudad_limpia'] = df_limpio['city'].astype(str).str.lower().str.strip()
df_rescate['ciudad_limpia'] = df_rescate['city'].astype(str).str.lower().str.strip()

# Cruce de datos y limpieza de duplicados
df_rescate = df_rescate.drop_duplicates(subset=['fecha_corta', 'ciudad_limpia'])
df_final = pd.merge(df_limpio, df_rescate[['fecha_corta', 'ciudad_limpia', 'shape', 'duration (seconds)']], 
                     on=['fecha_corta', 'ciudad_limpia'], how='left')

# Conversión de duración a numérico para análisis estadístico en SAS
df_final['duration (seconds)'] = pd.to_numeric(df_final['duration (seconds)'], errors='coerce')
df_final = df_final.drop(columns=['fecha_corta', 'ciudad_limpia'])

print("Cruce completado y variables recuperadas.")

Cruce completado y variables recuperadas.


## 4. Normalización de Datos y Casos Locales (Argentina)
En la etapa final, limpiamos el ruido en los nombres de ciudades (códigos HTML y paréntesis) y realizamos un rescate exhaustivo de los casos pertenecientes a Argentina. Identificamos falsos positivos (como Santa Fe en USA) para asegurar que el análisis local sea veraz y etiquetamos correctamente la columna 'country'.

In [26]:
# 1. Limpieza estética de la columna City
df_final['city'] = df_final['city'].apply(lambda x: html.unescape(str(x)))
df_final['city'] = df_final['city'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

# 2. Identificación de casos en Argentina (Búsqueda por palabras clave y exclusión de falsos positivos)
keywords = 'argentina|buenos aires|cordoba|rosario|mendoza|tucuman|salta|bariloche|mar del plata|neuquen|santa fe'
mask_city = df_final['city'].astype(str).str.lower().str.contains(keywords, na=False)
mask_state = df_final['state'].astype(str).str.lower().str.contains(keywords, na=False)
mask_country = df_final['country'].astype(str).str.lower().isin(['ar', 'argentina'])

# Exclusión de países anglosajones y el estado de New Mexico (NM)
paises_anglo = ['us', 'ca', 'gb', 'au', 'nz']
mask_falsos_positivos = (df_final['country'].astype(str).str.lower().isin(paises_anglo)) | (df_final['state'].astype(str).str.lower() == 'nm')

# Etiquetado final y exportación
mask_casos_reales = (mask_city | mask_state | mask_country) & ~mask_falsos_positivos
df_final.loc[mask_casos_reales, 'country'] = 'ar'

df_final.to_csv("ufo_sas_viya_definitivo_FINAL.csv", index=False)

print(f"Proceso ETL terminado. Casos finales: {len(df_final)}")
print(f"Casos confirmados en Argentina: {len(df_final[df_final['country'] == 'ar'])}")

Proceso ETL terminado. Casos finales: 192777
Casos confirmados en Argentina: 31


In [27]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 192777 entries, 0 to 192776
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   t_utc               192777 non-null  datetime64[us]
 1   lat                 192777 non-null  float64       
 2   lon                 192777 non-null  float64       
 3   city                192777 non-null  str           
 4   state               192777 non-null  str           
 5   country             192777 non-null  str           
 6   cluster_id          192777 non-null  int64         
 7   moon_illum          192777 non-null  float64       
 8   moon_alt_deg        192777 non-null  float64       
 9   nearest_airport_km  192777 non-null  float64       
 10  wx_bucket           192777 non-null  str           
 11  shape               186217 non-null  str           
 12  duration (seconds)  188662 non-null  float64       
dtypes: datetime64[us](1), float64(6), int64(

## 5.  Auditoría de Nulos

In [28]:
# ==========================================
# PASO 5.1: AUDITORÍA DE DATOS NULOS
# ==========================================
print("Análisis de integridad del cruce:")

# Calculamos el porcentaje de datos rescatados vs nulos
total_filas = len(df_final)
nulos_shape = df_final['shape'].isna().sum()
nulos_duration = df_final['duration (seconds)'].isna().sum()

print(f"- Filas totales: {total_filas}")
print(f"- Nulos en 'shape': {nulos_shape} ({round((nulos_shape/total_filas)*100, 2)}%)")
print(f"- Nulos en 'duration': {nulos_duration} ({round((nulos_duration/total_filas)*100, 2)}%)")

# REVISIÓN: ¿Los nulos de Argentina están controlados?
nulos_argentina = df_final[df_final['country'] == 'ar']['shape'].isna().sum()
print(f"- Casos de Argentina sin 'shape': {nulos_argentina}")

# OPCIÓN: Ver una muestra de los casos que no cruzaron
print("\nMuestra de casos sin 'shape' (posibles fallas de coincidencia en el Join):")
display(df_final[df_final['shape'].isna()][['t_utc', 'city', 'country']].head(5))

Análisis de integridad del cruce:
- Filas totales: 192777
- Nulos en 'shape': 6560 (3.4%)
- Nulos en 'duration': 4115 (2.13%)
- Casos de Argentina sin 'shape': 3

Muestra de casos sin 'shape' (posibles fallas de coincidencia en el Join):


,t_utc,city,country
96,2011-10-10 19:30:00,murfeesboro/smyrna,NAN
168,2006-10-12 00:00:00,rome,US
216,2011-10-11 21:00:00,monroe,US
323,2006-10-01 19:48:00,austin,US
336,2006-10-01 23:00:00,cuchara,US


Conclusión de Integridad:
"Se observa una tasa de recuperación superior al 96% para las variables rescatadas (shape y duration). El remanente de datos nulos (~3%) se atribuye a inconsistencias menores en la nomenclatura de ciudades compuestas (ej. ciudades separadas por barras) y a la ventana temporal del dataset histórico de Kaggle. Dado que el volumen de datos íntegros supera los 186.000 registros, la representatividad estadística para el análisis en SAS Viya está plenamente garantizada. En el caso específico de Argentina, solo 3 registros carecen de morfología, lo cual no impide un análisis local robusto."

Reemplazamos la palabra NAN por unknown en la columna Shape

In [29]:
# Rellenamos los nulos de forma con 'unknown' para mejorar la estética en los reportes
df_final['shape'] = df_final['shape'].fillna('unknown')

# La duración la dejamos con nulos (NaN) para que SAS no promedie ceros falsos

## 6.  Exportación

In [30]:
# Exportación del Dataset Maestro
nombre_archivo_final = "ufo_sas_viya_definitivo_FINAL.csv"
df_final.to_csv(nombre_archivo_final, index=False)

print(f"--- REPORTE DE CIERRE ---")
print(f"Archivo generado: {nombre_archivo_final}")
print(f"Peso estimado: ~17.8 MB")
print(f"Registros totales: {len(df_final)}")
print(f"Casos Argentina (etiquetados como 'ar'): {len(df_final[df_final['country'] == 'ar'])}")
print(f"-------------------------")
print("🚀 ¡ETL FINALIZADO! El archivo está listo para SAS Viya.")

--- REPORTE DE CIERRE ---
Archivo generado: ufo_sas_viya_definitivo_FINAL.csv
Peso estimado: ~17.8 MB
Registros totales: 192777
Casos Argentina (etiquetados como 'ar'): 31
-------------------------
🚀 ¡ETL FINALIZADO! El archivo está listo para SAS Viya.
